In [1]:
# 08_confound_inference.ipynb
# =============================================================================
# Robustness of the RQ2 decomposition to three reviewer concerns.
#
# (A) PERSON-NODE CONFOUND. In SMEsD most edges route through person nodes
#     (10,550 of 13,336), so the contagion GNN also ingests metapath2vec person
#     embeddings that the intra-only MLP never sees. Part of the intra->contagion
#     gain could therefore be static person features rather than firm-to-firm
#     contagion. We decompose the gain with five configurations:
#       1. intra                : MLP on 23-d company features.
#       2. intra + person feats : MLP on 23-d + pooled metapath2vec embeddings of
#                                 directly-connected persons (static, no propagation).
#       3. contagion cc-only    : GNN over company-company edges within 0-11 only
#                                 (no person pathway at all).
#       4. contagion 0-11       : GNN over all economic ties (person-mediated + direct).
#       5. contagion 0-19       : GNN over all relations.
#     This attributes the 0.081 uplift to (static person features) vs
#     (firm-to-firm contagion) vs (person-mediated contagion).
#
# (B) MULTI-SEED INFERENCE. DeLong and bootstrap are recomputed for every seed
#     (not seed 0 alone), reporting the distribution of p-values and CIs.
#
# (C) ARCHITECTURE-INVARIANCE. The intra vs contagion(0-11) decomposition is
#     repeated with the relation-attention operator to show the RQ2 conclusion
#     does not depend on the message-passing operator.
# 10 seeds, mask A (strict). Figures grayscale, dpi 600, png + pdf, no caption.
# =============================================================================
import warnings, pickle; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, norm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from torch_geometric.nn import HeteroConv, SAGEConv, GATConv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10})
sns.set_theme(style="whitegrid", palette="Greys_r")
SEEDS = list(range(10))


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


def ks_stat(y, s):
    fpr, tpr, _ = roc_curve(y, s); return float(np.max(tpr - fpr))


def metrics(y, s, thr=0.5):
    return {"AUC": roc_auc_score(y, s), "F1": f1_score(y, (s >= thr).astype(int),
            zero_division=0), "KS": ks_stat(y, s)}


def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    out = np.empty(N); out[J] = T; return out


def delong(y, s1, s2):
    pos = y == 1
    sx = np.vstack([s1[pos], s2[pos]]); sy = np.vstack([s1[~pos], s2[~pos]])
    m, n = sx.shape[1], sy.shape[1]
    tx = np.array([_midrank(sx[k]) for k in range(2)])
    ty = np.array([_midrank(sy[k]) for k in range(2)])
    txy = np.array([_midrank(np.concatenate([sx[k], sy[k]])) for k in range(2)])
    aucs = txy[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (txy[:, :m] - tx) / n; v10 = 1.0 - (txy[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    z = (aucs[0] - aucs[1]) / np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1] + 1e-12)
    return float(aucs[0]), float(aucs[1]), float(2 * (1 - norm.cdf(abs(z))))


def boot_ci(y, s, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]; vals = []
    for _ in range(n):
        bi = np.concatenate([rng.choice(pos, len(pos), True), rng.choice(neg, len(neg), True)])
        vals.append(roc_auc_score(y[bi], s[bi]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


# --- Load graph + build person-neighbor features -----------------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
N_COMP = comp.num_nodes
person_x = data["person"].x.numpy()
PDIM = person_x.shape[1]

# pooled metapath2vec embedding of each company's directly-connected persons
pers_sum = np.zeros((N_COMP, PDIM), dtype=np.float64)
pers_cnt = np.zeros(N_COMP, dtype=np.float64)
for et in data.edge_types:
    s, r, t = et
    ei = data[et].edge_index.numpy()
    if s == "company" and t == "person":
        for c, p in zip(ei[0], ei[1]):
            pers_sum[c] += person_x[p]; pers_cnt[c] += 1
    elif s == "person" and t == "company":
        for p, c in zip(ei[0], ei[1]):
            pers_sum[c] += person_x[p]; pers_cnt[c] += 1
pers_feat = np.divide(pers_sum, np.maximum(pers_cnt[:, None], 1)).astype(np.float32)
print(f"companies with >=1 person neighbor: {int((pers_cnt > 0).sum()):,} / {N_COMP:,}")

Xbase = comp.x.numpy().astype(np.float64)
Xbase[:, :2] = np.log1p(np.clip(Xbase[:, :2], 0, None))
y = comp.y
trm, vam, tem = comp.train_mask, comp.val_mask, comp.test_mask

ALL_ET = list(data.edge_types)
CC_0_11 = [et for et in ALL_ET if et[0] == "company" and et[2] == "company"
           and int(et[1].split("_")[1]) <= 11]
ET_0_11 = [et for et in ALL_ET if int(et[1].split("_")[1]) <= 11]
ET_0_19 = ALL_ET
print(f"company-company edge types within 0-11: {[et[1] for et in CC_0_11]}")


class HeteroGNN(nn.Module):
    def __init__(self, edge_types, in_dims, op="SAGE", hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.drop = dropout
        self.lin_in = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in in_dims.items()})
        def mk():
            if op == "SAGE":
                return {et: SAGEConv((hidden, hidden), hidden) for et in edge_types}
            return {et: GATConv((hidden, hidden), hidden, heads=1, add_self_loops=False)
                    for et in edge_types}
        self.convs = nn.ModuleList([HeteroConv(mk(), aggr="sum") for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, xd, eid):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in xd.items()}
        for conv in self.convs:
            h = conv(h, eid)
            h = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training)
                 for nt, v in h.items()}
        return self.head(h["company"])


class MLP(nn.Module):
    def __init__(self, d, hidden=64, out=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, out))

    def forward(self, x):
        return self.net(x)


def class_w():
    cls = torch.bincount(y[trm], minlength=2).float()
    return cls.sum() / (2 * cls.clamp(min=1))


def run(kind, seed, edge_types=None, op="SAGE", extra=None, return_prob=False):
    torch.manual_seed(seed); np.random.seed(seed)
    w = class_w()
    if kind == "mlp":
        feat = Xbase if extra is None else np.concatenate([Xbase, extra], 1)
        sc = StandardScaler().fit(feat[trm.numpy()])
        X = torch.tensor(sc.transform(feat).astype(np.float32))
        model = MLP(X.size(1)); fwd = lambda: model(X)
    else:
        Xc = StandardScaler().fit(Xbase[trm.numpy()]).transform(Xbase).astype(np.float32)
        xd = {"company": torch.tensor(Xc), "person": data["person"].x}
        in_dims = {"company": Xc.shape[1], "person": PDIM}
        eid = {et: data[et].edge_index for et in edge_types}
        model = HeteroGNN(edge_types, in_dims, op=op); fwd = lambda: model(xd, eid)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)
    best_val, best_state, wait = -1, None, 0
    for _ in range(200):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(fwd()[trm], y[trm], weight=w); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(fwd(), 1)[:, 1].numpy()
        va = roc_auc_score(y[vam].numpy(), p[vam.numpy()])
        if va > best_val:
            best_val, best_state, wait = va, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 30:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p = F.softmax(fwd(), 1)[:, 1].numpy()
    met = metrics(y[tem.numpy()].numpy(), p[tem.numpy()])
    return (met, p[tem.numpy()]) if return_prob else met


# --- (A) Confound decomposition: 5 configs x 10 seeds ------------------------
CONFIGS = [
    ("1_intra", "mlp", {}),
    ("2_intra_person", "mlp", {"extra": pers_feat}),
    ("3_contagion_cc_only", "gnn", {"edge_types": CC_0_11}),
    ("4_contagion_0_11", "gnn", {"edge_types": ET_0_11}),
    ("5_contagion_0_19", "gnn", {"edge_types": ET_0_19}),
]
rows = []
probs = {c[0]: [] for c in CONFIGS}
for seed in SEEDS:
    for name, kind, kw in CONFIGS:
        met, p = run(kind, seed, return_prob=True, **kw)
        rows.append({"config": name, "seed": seed, **met})
        probs[name].append(p)
raw = pd.DataFrame(rows)
raw.to_csv(TAB / "08_confound_raw.csv", index=False)
summary = raw.groupby("config").agg(AUC_mean=("AUC", "mean"), AUC_std=("AUC", "std"),
                                     F1_mean=("F1", "mean"), KS_mean=("KS", "mean")).reset_index().round(4)
summary.to_csv(TAB / "08_confound.csv", index=False)
print("=== (A) Person-confound decomposition (mask A, 10 seeds) ===")
print(summary.to_string(index=False))
m = summary.set_index("config")["AUC_mean"]
intra = m["1_intra"]
print(f"\n  intra                         {intra:.3f}")
print(f"  + static person features      {m['2_intra_person']:.3f}  (gain {m['2_intra_person']-intra:+.3f})")
print(f"  contagion company-company only{m['3_contagion_cc_only']:.3f}  (gain {m['3_contagion_cc_only']-intra:+.3f})")
print(f"  contagion 0-11 (full)         {m['4_contagion_0_11']:.3f}  (gain {m['4_contagion_0_11']-intra:+.3f})")
print(f"  -> firm-to-firm contagion alone adds {m['3_contagion_cc_only']-intra:+.3f}; "
      f"person-mediated paths add a further {m['4_contagion_0_11']-m['3_contagion_cc_only']:+.3f}; "
      f"static person features alone add {m['2_intra_person']-intra:+.3f}")

# --- (B) Multi-seed DeLong + bootstrap: intra vs cc-only and vs full ----------
yt = y[tem.numpy()].numpy()
infer_rows = []
for comp_name in ["3_contagion_cc_only", "4_contagion_0_11"]:
    for seed in SEEDS:
        a_i, a_c, p_del = delong(yt, probs["1_intra"][seed], probs[comp_name][seed])
        lo_c, hi_c = boot_ci(yt, probs[comp_name][seed], seed=seed)
        infer_rows.append({"comparison": f"intra_vs_{comp_name}", "seed": seed,
                           "AUC_intra": round(a_i, 4), "AUC_contagion": round(a_c, 4),
                           "delong_p": p_del, "boot_lo": round(lo_c, 4), "boot_hi": round(hi_c, 4)})
infer = pd.DataFrame(infer_rows)
infer.to_csv(TAB / "08_multiseed_inference.csv", index=False)
print("\n=== (B) Multi-seed DeLong (intra vs contagion), mask A ===")
for cn in ["3_contagion_cc_only", "4_contagion_0_11"]:
    sub = infer[infer.comparison == f"intra_vs_{cn}"]
    print(f"  {cn:22s}: DeLong p median={sub.delong_p.median():.2e} "
          f"max={sub.delong_p.max():.2e}  (significant at 0.05 in {int((sub.delong_p<0.05).sum())}/10 seeds)")

# --- (C) Architecture-invariance: attention decomposition --------------------
attn_rows = []
for seed in SEEDS:
    met_c = run("gnn", seed, edge_types=ET_0_11, op="ATTN")
    attn_rows.append({"config": "contagion_0_11_ATTN", "seed": seed, **met_c})
attn = pd.DataFrame(attn_rows)
attn_auc = attn["AUC"].mean()
intra_seeds = raw[raw.config == "1_intra"].sort_values("seed")["AUC"].values
attn_seeds = attn.sort_values("seed")["AUC"].values
t_a, p_a = ttest_rel(attn_seeds, intra_seeds)
attn.to_csv(TAB / "08_attention_decomposition.csv", index=False)
print(f"\n=== (C) Decomposition under relation-attention operator ===")
print(f"  intra {intra:.3f} -> contagion(0-11, attention) {attn_auc:.3f} "
      f"(gain {attn_auc-intra:+.3f}, paired t={t_a:.2f}, p={p_a:.2e})")
print(f"  SAGE gave {m['4_contagion_0_11']:.3f}; the contagion conclusion holds for both operators.")

# --- Figure: confound decomposition ------------------------------------------
fig, ax = plt.subplots(figsize=(6.4, 3.6))
order = ["1_intra", "2_intra_person", "3_contagion_cc_only", "4_contagion_0_11", "5_contagion_0_19"]
labels = ["Intra\n(MLP)", "Intra +\nperson feats", "Contagion\ncompany-only",
          "Contagion\n0-11 (full)", "Contagion\n0-19"]
means = [m[c] for c in order]
stds = [summary.set_index("config").loc[c, "AUC_std"] for c in order]
shades = ["#999999", "#777777", "#555555", "#333333", "#444444"]
ax.bar(np.arange(5), means, yerr=stds, color=shades, edgecolor="black", linewidth=0.5,
       error_kw={"elinewidth": 0.9, "capsize": 3}, width=0.62)
ax.axhline(intra, color="black", linestyle=":", linewidth=0.8)
ax.set_xticks(np.arange(5)); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("Test AUC (mask A)"); ax.set_ylim(0.6, 0.8)
savefig(fig, "fig_confound")

print("\n08_confound_inference complete: person-confound decomposition, multi-seed "
      "inference, architecture-invariance; 1 figure, 4 tables.")


companies with >=1 person neighbor: 3,137 / 3,976
company-company edge types within 0-11: ['rel_6', 'rel_7', 'rel_10', 'rel_11']


=== (A) Person-confound decomposition (mask A, 10 seeds) ===
             config  AUC_mean  AUC_std  F1_mean  KS_mean
            1_intra    0.6862   0.0083   0.6015   0.3119
     2_intra_person    0.6724   0.0142   0.6500   0.2615
3_contagion_cc_only    0.7234   0.0140   0.6264   0.3664
   4_contagion_0_11    0.7671   0.0136   0.6883   0.4357
   5_contagion_0_19    0.7658   0.0135   0.6723   0.4339

  intra                         0.686
  + static person features      0.672  (gain -0.014)
  contagion company-company only0.723  (gain +0.037)
  contagion 0-11 (full)         0.767  (gain +0.081)
  -> firm-to-firm contagion alone adds +0.037; person-mediated paths add a further +0.044; static person features alone add -0.014



=== (B) Multi-seed DeLong (intra vs contagion), mask A ===
  3_contagion_cc_only   : DeLong p median=2.12e-02 max=4.56e-01  (significant at 0.05 in 6/10 seeds)
  4_contagion_0_11      : DeLong p median=1.74e-04 max=9.12e-03  (significant at 0.05 in 10/10 seeds)



=== (C) Decomposition under relation-attention operator ===
  intra 0.686 -> contagion(0-11, attention) 0.743 (gain +0.056, paired t=8.11, p=1.99e-05)
  SAGE gave 0.767; the contagion conclusion holds for both operators.



08_confound_inference complete: person-confound decomposition, multi-seed inference, architecture-invariance; 1 figure, 4 tables.
